In [29]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [30]:
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("[WARN] ollama package not installed. Install with: pip install ollama")

In [31]:
DATA_DIR_CANDIDATES = [
    Path.cwd() / "CarHackingDataset",
    Path.cwd().parent / "CarHackingDataset",
]

DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    DATA_DIR = Path("CarHackingDataset")
    print(f"[WARN] CarHackingDataset not found in expected locations, defaulting to {DATA_DIR.resolve()}")
else:
    print(f"[INFO] Using datasets from {DATA_DIR.resolve()}")

DATASET_PATHS = {
    "DoS":   DATA_DIR / "DoS_dataset.csv",
    "Fuzzy": DATA_DIR / "Fuzzy_dataset.csv",
    "Gear":  DATA_DIR / "gear_dataset.csv",
    "RPM":   DATA_DIR / "RPM_dataset.csv",
}

WINDOW_SIZE = 200
WINDOW_STRIDE = 400  # increase stride to skip more windows
SAMPLE_RATIO = 0.02  # sample fewer windows
SHORT_ANSWERS_PER_WINDOW = 1  # minimal per-window questions
GLOBAL_SEED = 50

BYTE_COLUMNS = [f"Byte{i}" for i in range(1, 9)]
ATTACK_LABELS = ["DoS", "Fuzzy", "Gear", "RPM"]
SUPPRESSION_THRESHOLD = 1e-3
FLOODING_THRESHOLD = 5e-4
EXPECTED_COLUMNS = ["Timestamp", "ID", "DLC", *BYTE_COLUMNS, "Flag"]

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]  # enable all types

[INFO] Using datasets from C:\Users\Dub\Documents\GitHub\CAN_QA\CarHackingDataset


In [32]:
# LLM Configuration for Answer Enhancement
LLM_ENABLED = False  # disable for speed; set True when you want LLM answers
LLM_MODEL = "mistral"  # Change to your preferred model (e.g., "llama2", "neural-chat", "mistral")
LLM_BASE_URL = "http://localhost:11434"  # Default Ollama URL
MAX_ANSWER_TOKENS = 150  # Limit answer length
LLM_TEMPERATURE = 0.7  # Creativity level (0=deterministic, 1=creative)

if LLM_ENABLED:
    print("[INFO] LLM enhancement ENABLED")
    print(f"[INFO] Using model: {LLM_MODEL}")
    print(f"[INFO] Ollama URL: {LLM_BASE_URL}")
    print("[INFO] Make sure Ollama is running: ollama serve")
else:
    print("[WARN] LLM enhancement DISABLED - ollama package not available or turned off")


[WARN] LLM enhancement DISABLED - ollama package not available or turned off


In [33]:
def _normalize_flag_series(series: pd.Series) -> pd.Series:
    mapped = series.map({"R": 0, "T": 1})
    numeric = pd.to_numeric(series, errors="coerce")
    combined = mapped.fillna(numeric).fillna(0).astype(int)
    return combined


def _to_int_hex(val) -> int:
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s.lower().startswith("0x"):
        s = s[2:]
    try:
        return int(s, 16)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def load_datasets(paths: Dict[str, Path]):
    datasets: Dict[str, pd.DataFrame] = {}
    profiles: Dict[str, dict] = {}

    for name, path in paths.items():
        csv_path = Path(path)
        if not csv_path.exists():
            print(f"[WARN] Dataset {name} not found at {csv_path}, skipping.")
            continue

        df = pd.read_csv(csv_path)
        if not set(EXPECTED_COLUMNS).issubset(df.columns):
            # Raw CarHackingDataset files ship without headers; add them when missing
            df = pd.read_csv(csv_path, header=None, names=EXPECTED_COLUMNS)
        else:
            df = df.rename(columns=str)

        missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
        for col in missing_cols:
            df[col] = 0
        df = df[EXPECTED_COLUMNS]

        df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
        df["ID"] = df["ID"].apply(_to_int_hex)
        df["DLC"] = df["DLC"].apply(_to_int_hex)
        for col in BYTE_COLUMNS:
            df[col] = df[col].apply(_to_int_hex)

        df["Flag"] = _normalize_flag_series(df["Flag"])
        df = df.dropna(subset=["Timestamp", "ID"]).reset_index(drop=True)

        id_counts = df["ID"].value_counts()
        expected_ids = set(int(x) for x in id_counts.head(5).index.tolist())
        critical_ids = set(int(x) for x in id_counts.head(3).index.tolist())

        datasets[name] = df
        profiles[name] = {
            "expected_ids": expected_ids,
            "critical_ids": critical_ids,
            "attack_label": name,
        }

        print(f"[INFO] Loaded {name}: {len(df)} rows, "
              f"{len(expected_ids)} expected IDs, {len(critical_ids)} critical IDs.")

    return datasets, profiles


datasets, profiles = load_datasets({k: v for k, v in DATASET_PATHS.items() if k in SELECTED_DATASETS})
rng_global = np.random.default_rng(GLOBAL_SEED)


# Cell 3: helpers (window, stats)
def iter_window_starts(num_rows: int) -> List[int]:
    if num_rows < WINDOW_SIZE:
        return []
    return list(range(0, num_rows - WINDOW_SIZE + 1, WINDOW_STRIDE))


def sample_window_indices(starts: List[int], rng: np.random.Generator) -> List[int]:
    if not starts:
        return []
    sample_size = max(1, int(len(starts) * SAMPLE_RATIO))
    sample_size = min(sample_size, len(starts))
    return sorted(rng.choice(starts, size=sample_size, replace=False))


def format_window(df: pd.DataFrame) -> str:
    rows = []
    for _, row in df.iterrows():
        byte_vals = [int(row[col]) for col in BYTE_COLUMNS]
        rows.append(
            f"Timestamp={row['Timestamp']:.6f} | "
            f"ID={int(row['ID'])} | DLC={int(row['DLC'])} | "
            f"bytes={byte_vals} | Flag={int(row['Flag'])} |"
        )
    return "\n".join(rows)


def compute_basic_stats(df: pd.DataFrame) -> dict:
    stats = {}
    total_frames = len(df)
    if total_frames == 0:
        return stats

    id_counts = df["ID"].value_counts()
    stats["id_counts"] = id_counts
    stats["dominant_id"] = int(id_counts.index[0])
    stats["dominant_share"] = id_counts.iloc[0] / total_frames

    # timing
    if total_frames > 1:
        diffs = df["Timestamp"].to_numpy()[1:] - df["Timestamp"].to_numpy()[:-1]
        stats["diffs"] = diffs
        stats["gap_max"] = float(diffs.max())
        stats["gap_min"] = float(diffs.min())
        stats["gap_mean"] = float(diffs.mean())
        stats["gap_std"] = float(diffs.std())
        stats["window_duration"] = float(df["Timestamp"].iloc[-1] - df["Timestamp"].iloc[0])
    else:
        stats["diffs"] = np.array([])
        stats["gap_max"] = 0.0
        stats["gap_min"] = 0.0
        stats["gap_mean"] = 0.0
        stats["gap_std"] = 0.0
        stats["window_duration"] = 0.0

    # payload const for dominant id
    dom_group = df[df["ID"] == stats["dominant_id"]]
    if not dom_group.empty:
        payload = dom_group[BYTE_COLUMNS].to_numpy()
        stats["dominant_payload_var"] = float(payload.var())
    else:
        stats["dominant_payload_var"] = 0.0

    return stats


# Cell 4: Short Answer question generation
def generate_sa_dominant_id(df: pd.DataFrame, stats: dict) -> Optional[dict]:
    """
    Which CAN ID appears most frequently in this window?
    """
    dom_id = stats.get("dominant_id")
    if dom_id is None:
        return None
    
    id_counts = stats.get("id_counts")
    if id_counts is None or id_counts.empty:
        return None
    
    count = int(id_counts.iloc[0])
    total = int(id_counts.sum())
    
    return {
        "type": "dominant_id",
        "question": "Which CAN ID appears most frequently in this window?",
        "answer": f"0x{dom_id:03X}",
        "explanation": f"ID 0x{dom_id:03X} appears {count} times out of {total} total frames."
    }


def generate_sa_attack_type(stats: dict, profile: dict) -> Optional[dict]:
    """
    What attack type best fits this window?
    """
    gt_label = profile.get("attack_label", "DoS")
    return {
        "type": "attack_type",
        "question": "What attack type best fits the behavior in this window?",
        "answer": gt_label,
        "explanation": f"This window is labeled as {gt_label} attack."
    }


def generate_sa_timing_pattern(stats: dict) -> Optional[dict]:
    """
    Describe the timing behavior in this window.
    """
    gap_mean = stats.get("gap_mean", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    gap_max = stats.get("gap_max", 0.0)
    
    if gap_mean < FLOODING_THRESHOLD and gap_std < FLOODING_THRESHOLD:
        pattern = "Flooding - extremely small gaps with very consistent timing"
    elif gap_max > SUPPRESSION_THRESHOLD * 5:
        pattern = "Suppression - several large gaps indicating dropped or delayed frames"
    elif gap_std < gap_mean * 0.1:
        pattern = "Uniform - timing is mostly consistent with minimal jitter"
    else:
        pattern = "Irregular - timing varies unpredictably"
    
    return {
        "type": "timing_pattern",
        "question": "Describe the timing behavior observed in this window.",
        "answer": pattern,
        "explanation": f"Mean gap: {gap_mean:.6f}s, Std: {gap_std:.6f}s, Max gap: {gap_max:.6f}s"
    }


def generate_sa_missing_id(df: pd.DataFrame, profile: dict) -> Optional[dict]:
    """
    Is any expected control ID missing from this window?
    """
    expected_ids: Set[int] = profile.get("expected_ids", set())
    if not expected_ids:
        return None
    
    present_ids = set(int(x) for x in df["ID"].unique())
    missing = [eid for eid in expected_ids if eid not in present_ids]
    
    if missing:
        missing_str = ", ".join([f"0x{eid:03X}" for eid in missing])
        answer = f"Yes, {missing_str}"
    else:
        answer = "No, all expected control IDs are present"
    
    return {
        "type": "expected_id_missing",
        "question": "Is any expected control ID missing from this window?",
        "answer": answer,
        "explanation": f"Expected IDs: {', '.join([f'0x{x:03X}' for x in expected_ids])}, Present: {', '.join([f'0x{x:03X}' for x in present_ids])}"
    }


def generate_sa_dlc_summary(df: pd.DataFrame) -> Optional[dict]:
    """
    Summarize the DLC (Data Length Code) distribution in this window.
    """
    if "DLC" not in df.columns:
        return None
    
    dlc = df["DLC"].to_numpy()
    if dlc.size == 0:
        return None
    
    mean_dlc = float(dlc.mean())
    unique_dlcs = sorted(np.unique(dlc))
    most_common_dlc = int(np.argmax(np.bincount(dlc.astype(int))))
    
    summary = f"DLC values range from {int(dlc.min())} to {int(dlc.max())}, with mean {mean_dlc:.2f} and most common value {most_common_dlc}"
    
    return {
        "type": "dlc_summary",
        "question": "Summarize the DLC (Data Length Code) distribution in this window.",
        "answer": summary,
        "explanation": f"DLC statistics: min={int(dlc.min())}, max={int(dlc.max())}, mean={mean_dlc:.2f}, mode={most_common_dlc}"
    }


def generate_sa_payload_variance(df: pd.DataFrame, stats: dict) -> Optional[dict]:
    """
    What does the payload variance suggest about this window?
    """
    dom_id = stats.get("dominant_id")
    if dom_id is None:
        return None
    
    var = stats.get("dominant_payload_var", 0.0)
    
    if var < 1e-3:
        interpretation = "The dominant ID transmits identical payloads repeatedly, suggesting constant sensor readings or a crafted/synthetic attack."
    elif var < 1.0:
        interpretation = "The dominant ID shows moderate payload variation, suggesting normal sensor fluctuations."
    else:
        interpretation = "The dominant ID shows high payload variance, suggesting significant variation in data or possible payload injection."
    
    return {
        "type": "payload_variance",
        "question": "What does the payload variance suggest about this window?",
        "answer": interpretation,
        "explanation": f"Dominant ID 0x{dom_id:03X} has payload variance: {var:.6f}"
    }


def generate_sa_irregular_id(df: pd.DataFrame, rng: np.random.Generator) -> Optional[dict]:
    """
    Which ID shows the most irregular timing pattern?
    """
    gaps_std = {}
    for id_val, group in df.groupby("ID"):
        ts = group["Timestamp"].to_numpy()
        if ts.size < 3:
            continue
        diffs = np.diff(ts)
        if diffs.size == 0:
            continue
        gaps_std[int(id_val)] = float(diffs.std())
    
    if not gaps_std:
        return None
    
    sorted_ids = sorted(gaps_std.items(), key=lambda x: x[1], reverse=True)
    correct_id = sorted_ids[0][0]
    std_val = sorted_ids[0][1]
    
    return {
        "type": "id_irregular_timing",
        "question": "Which ID shows the most irregular timing pattern?",
        "answer": f"0x{correct_id:03X}",
        "explanation": f"ID 0x{correct_id:03X} has the highest timing standard deviation: {std_val:.6f}s"
    }


def generate_sa_burst_explanation(stats: dict) -> Optional[dict]:
    """
    What best explains the burst behavior observed?
    """
    dom_share = stats.get("dominant_share", 0.0)
    gap_mean = stats.get("gap_mean", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    
    if gap_mean < FLOODING_THRESHOLD and dom_share > 0.5:
        explanation = "Flooding from a compromised ECU - very small inter-frame gaps with high ID dominance"
    elif gap_mean > SUPPRESSION_THRESHOLD and gap_std > gap_mean * 0.5:
        explanation = "Diagnostic or recovery traffic causing bursts - large gaps with high variance"
    else:
        explanation = "Normal periodic behavior with minor bursts - typical timing patterns"
    
    return {
        "type": "burst_explanation",
        "question": "What best explains the burst behavior observed in this window?",
        "answer": explanation,
        "explanation": f"Dominant share: {dom_share:.2%}, Mean gap: {gap_mean:.6f}s, Gap std: {gap_std:.6f}s"
    }


def generate_sa_window_characterization(stats: dict) -> Optional[dict]:
    """
    How would you characterize this window overall?
    """
    dom_share = stats.get("dominant_share", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    gap_mean = stats.get("gap_mean", 0.0)
    window_duration = stats.get("window_duration", 0.0)
    
    if dom_share > 0.7 and gap_std < FLOODING_THRESHOLD:
        characterization = "Highly irregular and unsafe - dominated by a single ID with very consistent flooding"
    elif dom_share < 0.4 and gap_std < gap_mean * 0.1:
        characterization = "Uniform and typical - well-distributed across IDs with consistent timing"
    else:
        characterization = "Mostly stable with minor anomalies - some irregularities but generally normal"
    
    return {
        "type": "overall_window",
        "question": "How would you characterize this window overall?",
        "answer": characterization,
        "explanation": f"Duration: {window_duration:.3f}s, Dominant share: {dom_share:.2%}, ID count: {len(stats.get('id_counts', []))}"
    }

[INFO] Loaded DoS: 3665771 rows, 5 expected IDs, 3 critical IDs.
[INFO] Loaded Fuzzy: 3838860 rows, 5 expected IDs, 3 critical IDs.
[INFO] Loaded Gear: 4443142 rows, 5 expected IDs, 3 critical IDs.
[INFO] Loaded RPM: 4621702 rows, 5 expected IDs, 3 critical IDs.


In [34]:
# LLM Integration for Answer Enhancement
def enhance_answer_with_llm(question: str, rule_based_answer: str, 
                            explanation: str, context_preview: str) -> Optional[str]:
    """
    Use Ollama to generate a more comprehensive answer based on rule-based answer.
    Returns the LLM-generated answer or None if LLM is unavailable.
    """
    if not LLM_ENABLED:
        return None
    
    # Truncate context to avoid token limits
    context_preview = context_preview[:500] if context_preview else "N/A"
    
    prompt = f"""You are a CAN network security analyst. Given a question about CAN bus data analysis and an initial rule-based answer, 
provide a concise, comprehensive answer that builds upon the rule-based answer.

Question: {question}

Rule-based Answer: {rule_based_answer}

Data Context (first few lines):
{context_preview}

Provide a clear, detailed answer (max {MAX_ANSWER_TOKENS} words) that refines or expands on the rule-based answer. Be technical but accessible."""

    try:
        response = ollama.generate(
            model=LLM_MODEL,
            prompt=prompt,
            stream=False,
            options={
                "temperature": LLM_TEMPERATURE,
                "num_predict": MAX_ANSWER_TOKENS,
            }
        )
        
        llm_answer = response.get("response", "").strip()
        if llm_answer:
            # Clean up the answer
            llm_answer = llm_answer.replace("\n", " ")
            # Limit to max tokens as a safety measure
            words = llm_answer.split()[:MAX_ANSWER_TOKENS]
            llm_answer = " ".join(words)
            return llm_answer
    except Exception as e:
        print(f"[WARN] LLM call failed: {e}")
        return None
    
    return None


def enhance_sa_with_llm(sa: dict, context_preview: str) -> dict:
    """
    Enhance a short answer question with LLM-generated answer.
    Stores both rule-based and LLM answers in the question object.
    """
    if not LLM_ENABLED:
        return sa
    
    llm_answer = enhance_answer_with_llm(
        question=sa["question"],
        rule_based_answer=sa["answer"],
        explanation=sa.get("explanation", ""),
        context_preview=context_preview
    )
    
    if llm_answer:
        sa["llm_answer"] = llm_answer
    
    return sa


In [35]:
# ==== Optional: enrich profiles with global baseline ID rates ====
# Run once after datasets, profiles are created.
for name, df_full in datasets.items():
    id_counts_full = df_full["ID"].value_counts()
    total_full = len(df_full)
    baseline = {int(i): float(c / total_full) for i, c in id_counts_full.items()}
    profiles[name]["baseline_id_rate"] = baseline


def generate_sa_for_window(df: pd.DataFrame, profile: dict, rng: np.random.Generator) -> List[dict]:
    """Generate short answer questions for a single window"""
    stats = compute_basic_stats(df)
    sas: List[dict] = []

    # Attack type
    sa_attack = generate_sa_attack_type(stats, profile)
    if sa_attack:
        sas.append(sa_attack)

    # Dominant ID
    sa_dom = generate_sa_dominant_id(df, stats)
    if sa_dom:
        sas.append(sa_dom)

    # Timing pattern
    sa_timing = generate_sa_timing_pattern(stats)
    if sa_timing:
        sas.append(sa_timing)

    # Missing expected ID
    sa_missing = generate_sa_missing_id(df, profile)
    if sa_missing:
        sas.append(sa_missing)

    # DLC summary
    sa_dlc = generate_sa_dlc_summary(df)
    if sa_dlc:
        sas.append(sa_dlc)

    # Payload variance
    sa_payload = generate_sa_payload_variance(df, stats)
    if sa_payload:
        sas.append(sa_payload)

    # Irregular timing ID
    sa_irreg = generate_sa_irregular_id(df, rng)
    if sa_irreg:
        sas.append(sa_irreg)

    # Burst explanation
    sa_burst = generate_sa_burst_explanation(stats)
    if sa_burst:
        sas.append(sa_burst)

    # Overall characterization
    sa_overall = generate_sa_window_characterization(stats)
    if sa_overall:
        sas.append(sa_overall)

    # Limit per-window short answers
    if len(sas) > SHORT_ANSWERS_PER_WINDOW:
        idxs = rng.choice(len(sas), size=SHORT_ANSWERS_PER_WINDOW, replace=False)
        sas = [sas[i] for i in idxs]

    return sas


In [36]:
# No option shuffling needed for short answer questions

In [37]:
def generate_mcq_for_window(df: pd.DataFrame, profile: dict, rng: np.random.Generator) -> List[dict]:
    """Wrapper for backward compatibility"""
    return generate_sa_for_window(df, profile, rng)

In [38]:
# Cell 5: main loop – per-dataset short answer question generation with LLM enhancement
for ds_idx, (name, df) in enumerate(datasets.items()):
    print(f"[INFO] Generating short answer questions for {name}")
    starts = iter_window_starts(len(df))
    sampled_starts = sample_window_indices(starts, rng_global)

    out_dir = Path(f"{name}_sa_qa")
    out_dir.mkdir(parents=True, exist_ok=True)
    ql_path = out_dir / f"{name.lower()}_sa_questions.jsonl"
    qj_path = out_dir / f"{name.lower()}_sa_questions.json"

    ql_path.write_text("", encoding="utf-8")

    qa_id_counter = 0
    records_all = []

    with ql_path.open("a", encoding="utf-8") as f:
        for window_idx, start in enumerate(tqdm(sampled_starts, desc=f"{name} windows")):
            window = df.iloc[start:start + WINDOW_SIZE].copy().reset_index(drop=True)
            sa_items = generate_sa_for_window(window, profiles[name], rng_global)
            if not sa_items:
                continue

            context = format_window(window)
            context_preview = "\n".join(context.split("\n")[:5])  # First 5 lines for LLM
            
            for local_q_idx, sa in enumerate(sa_items):
                # Enhance answer with LLM if enabled
                if LLM_ENABLED:
                    sa = enhance_sa_with_llm(sa, context_preview)
                
                qa_id = f"{name}_SA_{window_idx:06d}_{local_q_idx:02d}"
                record = {
                    "qa_id": qa_id,
                    "metadata": {
                        "dataset": name,
                        "window_index": int(window_idx),
                        "window_start": int(start),
                        "window_size": int(WINDOW_SIZE),
                    },
                    "context": context,
                    "sa_type": sa["type"],
                    "question": sa["question"],
                    "answer": sa["answer"],  # Rule-based answer
                    "llm_answer": sa.get("llm_answer", None),  # LLM-enhanced answer
                    "explanation": sa.get("explanation", ""),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                records_all.append(record)
                qa_id_counter += 1

    with qj_path.open("w", encoding="utf-8") as jf:
        json.dump(records_all, jf, ensure_ascii=False, indent=2)

    print(f"[INFO] {name}: saved {qa_id_counter} short answer questions -> {ql_path}, {qj_path}")


[INFO] Generating short answer questions for DoS


DoS windows: 100%|██████████| 183/183 [00:03<00:00, 58.27it/s]


[INFO] DoS: saved 183 short answer questions -> DoS_sa_qa\dos_sa_questions.jsonl, DoS_sa_qa\dos_sa_questions.json
[INFO] Generating short answer questions for Fuzzy


Fuzzy windows: 100%|██████████| 191/191 [00:03<00:00, 54.67it/s]


[INFO] Fuzzy: saved 191 short answer questions -> Fuzzy_sa_qa\fuzzy_sa_questions.jsonl, Fuzzy_sa_qa\fuzzy_sa_questions.json
[INFO] Generating short answer questions for Gear


Gear windows: 100%|██████████| 222/222 [00:03<00:00, 58.47it/s]


[INFO] Gear: saved 222 short answer questions -> Gear_sa_qa\gear_sa_questions.jsonl, Gear_sa_qa\gear_sa_questions.json
[INFO] Generating short answer questions for RPM


RPM windows: 100%|██████████| 231/231 [00:04<00:00, 57.01it/s]

[INFO] RPM: saved 231 short answer questions -> RPM_sa_qa\rpm_sa_questions.jsonl, RPM_sa_qa\rpm_sa_questions.json


In [39]:
# Quick test of LLM integration
if OLLAMA_AVAILABLE:
    test_question = "What is 2+2?"
    test_answer = "4"
    test_llm_answer = enhance_answer_with_llm(
        question=test_question,
        rule_based_answer=test_answer,
        explanation="Basic math",
        context_preview="Test context"
    )
    print("LLM Test Result:")
    print(f"Question: {test_question}")
    print(f"Rule-based: {test_answer}")
    print(f"LLM Answer: {test_llm_answer}")
else:
    print("Ollama not available!")

LLM Test Result:
Question: What is 2+2?
Rule-based: 4
LLM Answer: None
